# Task 3: Consultas analíticas e dashboard

Este notebook se conecta ao Amazon Athena para consultar os dados já transformados em esquema estrela e salvos no S3 através do AWS Glue.

In [1]:
import awswrangler as wr
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Variável de configuração com o nome do database no Glue/Athena
GLUE_DATABASE = "classicmodels_analytics"
wr.config.workgroup = "classicmodels-analytics-workgroup"

### 4.2 - Consulta exploratória em dimensão
Consulta simples para inspecionar os dados carregados no S3 via Glue.

In [2]:
df_products = wr.athena.read_sql_query("""
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
""", database=GLUE_DATABASE)

display(df_products.head())

,product_id,product_name,product_line,product_vendor
0,S12_1666,1958 Setra Bus,Trucks and Buses,Welly Diecast Productions
1,S12_3990,1970 Plymouth Hemi Cuda,Classic Cars,Studio M Art Models
2,S12_4473,1957 Chevy Pickup,Trucks and Buses,Exoto Designs
3,S12_4675,1969 Dodge Charger,Classic Cars,Welly Diecast Productions
4,S18_1589,1965 Aston Martin DB5,Classic Cars,Classic Metal Creations


### 4.3 - Vendas totais por país
Calcula o total de vendas por país combinando a fato e a dimensão.

In [3]:
df_country_sales = wr.athena.read_sql_query("""
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
""", database=GLUE_DATABASE)

display(df_country_sales)

,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


### 4.4 - Detalhamento por data, linha de produto, produto e país
Base analítica detalhada que será utilizada no dashboard interativo.

In [4]:
df_detailed = wr.athena.read_sql_query("""
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
""", database=GLUE_DATABASE)

df_detailed['full_date'] = pd.to_datetime(df_detailed['full_date'])
display(df_detailed.head())

,full_date,product_line,product_name,country,total_sales
0,2003-01-29,Vintage Cars,1939 Cadillac Limousine,Norway,1670.75
1,2003-01-29,Vintage Cars,18th Century Vintage Horse Carriage,Norway,2173.00
2,2003-01-29,Vintage Cars,1934 Ford V8 Coupe,Norway,2164.40
3,2003-01-29,Trucks and Buses,1940 Ford Pickup Truck,Norway,3307.50
4,2003-01-29,Trucks and Buses,1958 Setra Bus,Norway,3284.28


### 4.5 - Dashboard interativo
Painel dinâmico (Plotly FigureWidget)

In [ ]:
countries = ['Todos'] + sorted(df_detailed['country'].dropna().unique().tolist())
product_lines = ['Todas'] + sorted(df_detailed['product_line'].dropna().unique().tolist())
min_date = df_detailed['full_date'].min().date() if not df_detailed.empty else None

# Controles UI
w_country = widgets.Dropdown(options=countries, value='Todos', description='País:')
w_product_line = widgets.Dropdown(options=product_lines, value='Todas', description='Linha Prod:')
w_dates = widgets.DatePicker(description='Data Inicial:', value=min_date)
w_top_n = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Top N:')

# Gráfico do Plotly desenhado como um Widget nativo do Jupyter
fig = go.FigureWidget(
    layout=go.Layout(
        title='Top Produtos em Vendas',
        xaxis_title='Vendas Totais ($)',
        yaxis_title='Produto',
        margin=dict(l=0, r=0, t=40, b=0)
    )
)

def update_dashboard(*args):
    df_filtered = df_detailed.copy()
    
    # Aplica os filtros dos widgets atuais
    if w_dates.value:
        df_filtered = df_filtered[df_filtered['full_date'] >= pd.Timestamp(w_dates.value)]
    if w_country.value != 'Todos':
        df_filtered = df_filtered[df_filtered['country'] == w_country.value]
    if w_product_line.value != 'Todas':
        df_filtered = df_filtered[df_filtered['product_line'] == w_product_line.value]
    
    if df_filtered.empty:
        with fig.batch_update():
            fig.data = []
        return
    
    df_agg = df_filtered.groupby('product_name')['total_sales'].sum().reset_index()
    df_agg = df_agg.sort_values(by='total_sales', ascending=False).head(w_top_n.value)
    
    # Atualiza o gráfico de forma atômica
    with fig.batch_update():
        fig.data = []
        fig.add_bar(
            x=df_agg['total_sales'], 
            y=df_agg['product_name'], 
            orientation='h',
            marker=dict(
                color=df_agg['total_sales'], 
                colorscale='Viridis'
            )
        )
        fig.layout.yaxis.categoryorder = 'total ascending'

# Monitora mudanças de valores em todos os controles
w_country.observe(update_dashboard, 'value')
w_product_line.observe(update_dashboard, 'value')
w_dates.observe(update_dashboard, 'value')
w_top_n.observe(update_dashboard, 'value')

# Renderiza primeira vez
update_dashboard()

ui_controls = widgets.VBox([w_country, w_product_line, w_dates, w_top_n])
ui = widgets.VBox([ui_controls, fig])

display(ui)